In [2]:
import pandas as pd

df = pd.read_csv("../data/wilde_fire_data_county.csv")

In [6]:
df.value_counts("incident_county")

incident_county
Riverside                                       339
San Diego                                       184
Fresno                                          179
Kern                                            171
San Luis Obispo                                 125
                                               ... 
Los Angeles, San Bernardino                       1
Orange, Riverside                                 1
Alameda, Humboldt                                 1
Fresno, Monterey                                  1
Calaveras, San Joaquin, Stanislaus, Tuolumne      1
Name: count, Length: 107, dtype: int64

In [36]:
import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster

df["incident_date_created"] = pd.to_datetime(
    df["incident_date_created"], errors="coerce"
)

ca_incidents = df[
    ~((df["incident_longitude"] == 1.0) & (df["incident_latitude"] == 1.0))
    & (df["incident_longitude"].between(-125.0, -114.0))
    & (df["incident_latitude"].between(32.0, 42.0))
    & (df["incident_date_created"].between("2025-01-01", "2025-12-31"))
].copy()

ca_incidents["log_acres"] = np.log1p(ca_incidents["incident_acres_burned"])

m = folium.Map(location=[37.2, -119.5], zoom_start=6, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)

for _, row in ca_incidents.iterrows():
    popup_text = f"""
    <b>Incident:</b> {row.get("incident_name", "Unknown")}<br>
    <b>Date:</b> {row["incident_date_created"].date() if pd.notnull(row["incident_date_created"]) else "Unknown"}<br>
    <b>Acres Burned:</b> {row.get("incident_acres_burned", "Unknown")}
    """

    folium.CircleMarker(
        location=[row["incident_latitude"], row["incident_longitude"]],
        radius=max(3, row["log_acres"] * 2),
        popup=popup_text,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.6,
    ).add_to(marker_cluster)

m.save("../results/california_wildfire_map_2025.html")
m


In [38]:
import numpy as np
import pandas as pd
import folium
from branca.colormap import linear

df["incident_date_created"] = pd.to_datetime(
    df["incident_date_created"], errors="coerce"
)

ca_incidents = df[
    ~((df["incident_longitude"] == 1.0) & (df["incident_latitude"] == 1.0))
    & (df["incident_longitude"].between(-125.0, -114.0))
    & (df["incident_latitude"].between(32.0, 42.0))
    & (df["incident_date_created"].between("2025-01-01", "2025-12-31"))
].copy()

print(f"Found {len(ca_incidents)} incidents in California")

# safer numeric conversion
ca_incidents["incident_acres_burned"] = pd.to_numeric(
    ca_incidents["incident_acres_burned"], errors="coerce"
)
ca_incidents = ca_incidents.dropna(
    subset=["incident_latitude", "incident_longitude", "incident_acres_burned"]
)

# log scale for nicer bubble sizing
ca_incidents["log_acres"] = np.log1p(ca_incidents["incident_acres_burned"])

# base map
m = folium.Map(location=[37.2, -119.5], zoom_start=6, tiles="CartoDB positron")

# color scale
colormap = linear.YlOrRd_09.scale(
    ca_incidents["incident_acres_burned"].min(),
    ca_incidents["incident_acres_burned"].max(),
)
colormap.caption = "Acres Burned"

for _, row in ca_incidents.iterrows():
    incident_name = row.get("incident_name", "Unknown")
    county = row.get("incident_county", "Unknown")
    acres = row["incident_acres_burned"]
    date = (
        row["incident_date_created"].strftime("%Y-%m-%d")
        if pd.notnull(row["incident_date_created"])
        else "Unknown"
    )

    popup_html = f"""
    <b>Incident:</b> {incident_name}<br>
    <b>Date:</b> {date}<br>
    <b>County:</b> {county}<br>
    <b>Acres Burned:</b> {acres:,.0f}
    """

    folium.CircleMarker(
        location=[row["incident_latitude"], row["incident_longitude"]],
        radius=max(3, row["log_acres"] * 1.8),
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{incident_name}: {acres:,.0f} acres",
        color=colormap(acres),
        fill=True,
        fill_color=colormap(acres),
        fill_opacity=0.7,
        weight=1,
    ).add_to(m)

colormap.add_to(m)
m.save("../results/california_wildfire_map_2025_bubble.html")
m

Found 558 incidents in California
